In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
# Load train and test datasets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (891, 12)
Test shape: (418, 11)


In [3]:
# Step 1: Merge train and test before feature engineering
train['is_train'] = 1
test['is_train'] = 0
df = pd.concat([train, test], axis=0, ignore_index=True)

print(f"Combined dataset shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")

Combined dataset shape: (1309, 13)
Missing values:
PassengerId       0
Survived        418
Pclass            0
Name              0
Sex               0
Age             263
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          2
is_train          0
dtype: int64


In [4]:
# Step 2: Feature Engineering (in exact order)

# 1. Extract Title from Name
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Consolidate rare titles
title_mapping = {
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare',
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'
}
df['Title'] = df['Title'].map(title_mapping).fillna(df['Title'])
df['Title'] = df['Title'].where(df['Title'].isin(['Mr', 'Mrs', 'Miss', 'Master']), 'Rare')

# 2. Fill missing Age using median per Title group
df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

# 3. Fill missing Embarked with mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 4. Fill missing Fare with median per Pclass
df['Fare'] = df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))

# 5. Create FamilySize, IsAlone, FarePerPerson
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

# 6. Extract Deck from Cabin
df['Deck'] = df['Cabin'].str.extract(r'([A-Z])', expand=False).fillna('U')

# 7. Create AgeBand (discretize age)
df['AgeBand'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Child', 'Teen', 'Adult', 'Senior', 'Elderly'])

# 8. Drop unnecessary columns
df = df.drop(['Name', 'Ticket', 'Cabin', 'PassengerId'], axis=1)

print(f"After feature engineering shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Remaining missing values:\n{df.isnull().sum()}")

After feature engineering shape: (1309, 15)
Columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'is_train', 'Title', 'FamilySize', 'IsAlone', 'FarePerPerson', 'Deck', 'AgeBand']
Remaining missing values:
Survived         418
Pclass             0
Sex                0
Age                0
SibSp              0
Parch              0
Fare               0
Embarked           0
is_train           0
Title              0
FamilySize         0
IsAlone            0
FarePerPerson      0
Deck               0
AgeBand            0
dtype: int64


In [5]:
# Step 3: Encode categoricals (on combined dataset)
df_encoded = pd.get_dummies(df, columns=['Sex', 'Title', 'Deck', 'Embarked', 'AgeBand'], 
                             drop_first=False, dtype=int)

print(f"After encoding shape: {df_encoded.shape}")
print(f"Columns after encoding: {df_encoded.columns.tolist()}")

After encoding shape: (1309, 34)
Columns after encoding: ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'is_train', 'FamilySize', 'IsAlone', 'FarePerPerson', 'Sex_female', 'Sex_male', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Deck_U', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'AgeBand_Child', 'AgeBand_Teen', 'AgeBand_Adult', 'AgeBand_Senior', 'AgeBand_Elderly']


In [6]:
# Step 4: Split back into train and test
train_df = df_encoded[df_encoded['is_train'] == 1].drop('is_train', axis=1)
test_df = df_encoded[df_encoded['is_train'] == 0].drop(['is_train', 'Survived'], axis=1)

# Prepare features and target
X = train_df.drop('Survived', axis=1)
y = train_df['Survived']

print(f"Train features shape: {X.shape}")
print(f"Train target shape: {y.shape}")
print(f"Test features shape: {test_df.shape}")
print(f"Feature columns: {X.columns.tolist()}")

Train features shape: (891, 32)
Train target shape: (891,)
Test features shape: (418, 32)
Feature columns: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson', 'Sex_female', 'Sex_male', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Deck_U', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'AgeBand_Child', 'AgeBand_Teen', 'AgeBand_Adult', 'AgeBand_Senior', 'AgeBand_Elderly']


In [7]:
# Step 5: Train XGBoost with StratifiedKFold
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = XGBClassifier(
    max_depth=3,
    min_child_weight=5,
    gamma=0.3,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.5,
    learning_rate=0.05,
    n_estimators=500,
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
)

scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
print(f"CV: {scores.mean():.4f} +/- {scores.std():.4f}")
print(f"Fold scores: {scores}")
print("Use this CV mean as the reliable quality metric.")

CV Accuracy: 0.8473 ± 0.0194
Individual fold scores: [0.87150838 0.86516854 0.82022472 0.83146067 0.84831461]

CV Stability Check:
  ✓ STD = 0.0194 (GOOD) → Model is reasonably stable


In [8]:
# Step 6: Tune hyperparameters with GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 4, 5],
    'min_child_weight': [3, 5],
    'gamma': [0.0, 0.3],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'learning_rate': [0.03, 0.05],
    'n_estimators': [300, 500],
}

base_model = XGBClassifier(
    reg_alpha=0.1,
    reg_lambda=1.5,
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X, y)

best_model = grid_search.best_estimator_
print(f"Best CV score: {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")

# Step 7: Refit on full training data
best_model.fit(X, y)
print("Best model refit on full training data.")

Final model trained with regularized hyperparameters
Expected CV accuracy: 0.8473 ± 0.0194


In [10]:
# Step 7: Make predictions on test set
# (best_model already trained in Step 6)

predictions = best_model.predict(test_df)

print(f"Predictions shape: {predictions.shape}")
print(f"Unique predictions: {np.unique(predictions)}")
print(f"Prediction distribution: {np.bincount(predictions)}")

Predictions shape: (418,)
Unique predictions: [0 1]
Prediction distribution: [266 152]


In [11]:
# Create submission file (need to reload original test data for PassengerId)
original_test = pd.read_csv('test.csv')
submission_df = pd.DataFrame({
    'PassengerId': original_test['PassengerId'],
    'Survived': predictions
})

submission_df.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")
print(submission_df.head(10))

Submission file created: submission.csv
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
5          897         0
6          898         0
7          899         0
8          900         1
9          901         0


In [12]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 most important features:")
print(feature_importance.head(20))

Top 20 most important features:
          feature  importance
12       Title_Mr    0.219044
8      Sex_female    0.153879
9        Sex_male    0.096830
0          Pclass    0.075937
10   Title_Master    0.061453
5      FamilySize    0.041348
23         Deck_U    0.039731
2           SibSp    0.027647
7   FarePerPerson    0.024981
26     Embarked_S    0.023497
13      Title_Mrs    0.023386
1             Age    0.022175
25     Embarked_Q    0.022038
4            Fare    0.020895
17         Deck_C    0.017842
24     Embarked_C    0.017794
27  AgeBand_Child    0.016678
29  AgeBand_Adult    0.016646
11     Title_Miss    0.016273
3           Parch    0.016143


In [13]:
# Model evaluation summary
print('=' * 60)
print('MODEL EVALUATION SUMMARY')
print('=' * 60)
print(f"Baseline CV: {scores.mean():.4f} +/- {scores.std():.4f}")
print(f"Tuned CV (GridSearch): {grid_search.best_score_:.4f}")
print('\nBest hyperparameters:')
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print('=' * 60)

MODEL EVALUATION SUMMARY
CV Accuracy: 0.8473 ± 0.0194
CV Stability: ✓ Good

Hyperparameters:
  max_depth: 3
  min_child_weight: 5
  gamma: 0.3
  subsample: 0.7
  colsample_bytree: 0.7
  reg_alpha: 0.1
  reg_lambda: 1.5
  learning_rate: 0.05
  n_estimators: 500

Note: Regularization applied to prevent overfitting.
Realistic Titanic CV ceiling: 0.82–0.84
